# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)


In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04*.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock [mm]": 170.0,
    "rear_shock [mm]": 165.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# ---- Batch orchestration ----

pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

logger.info(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    logger.info("  %s", p.name)

sessions = []
events_all = []
metrics_all = []

for p in CSV_FILES:
    logger.info(f"Processing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
    )

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()





INFO:__main__:Found 2 files:
INFO:__main__:  2026-01-04_06-37-32.CSV
INFO:__main__:  2026-01-04_07-08-37.CSV
INFO:__main__:Processing 2026-01-04_06-37-32.CSV ...
INFO:bodaqs_analysis.pipeline:Session load complete: C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04_06-37-32.CSV
INFO:bodaqs_analysis.pipeline:Session pre-process complete
INFO:bodaqs_analysis.pipeline:Schema load complete
INFO:bodaqs_analysis.detect:Built EVENTS_DF with 32 rows from 2 schema event(s) → 4 sensor-expanded event(s)
INFO:bodaqs_analysis.pipeline:Event detection complete
INFO:bodaqs_analysis.pipeline:events rows: 32
INFO:bodaqs_analysis.pipeline:Running segment extraction for schema events: ['deep_compression', 'rebounds']
INFO:bodaqs_analysis.pipeline:Segment extraction complete (schema_id=deep_compression)
INFO:bodaqs_analysis.pipeline:segments valid (schema_id=deep_compression): 0/0
INFO:bodaqs_analysis.pipeline:Metrics calculation complete (schema_id=deep_compression)
INFO:bodaqs_analysis.pipeline:Segm